# Data Exploration - Medical Multimodal Retrieval

This notebook explores the MIMIC-CXR dataset for medical image retrieval.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import os

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## Load Dataset

In [ ]:
# Paths
BASE_DIR = r"C:\Users\sagar\OneDrive\Desktop\IISC\data\mimic_cxr_project"
TRAIN_CSV = os.path.join(BASE_DIR, "processed", "train_processed.csv")
VAL_CSV = os.path.join(BASE_DIR, "processed", "val_processed.csv")
TEST_CSV = os.path.join(BASE_DIR, "processed", "test_processed.csv")

# Load data
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

print(f"Train samples: {len(train_df):,}")
print(f"Validation samples: {len(val_df):,}")
print(f"Test samples: {len(test_df):,}")

## Dataset Statistics

In [ ]:
# Basic statistics
print("Dataset Overview:")
print(f"Total samples: {len(train_df) + len(val_df) + len(test_df):,}")
print(f"Train/Val/Test split: {len(train_df)/(len(train_df)+len(val_df)+len(test_df))*100:.1f}% / {len(val_df)/(len(train_df)+len(val_df)+len(test_df))*100:.1f}% / {len(test_df)/(len(train_df)+len(val_df)+len(test_df))*100:.1f}%")

In [ ]:
# Text length analysis
train_df['caption_length'] = train_df['structured_caption'].str.len()
val_df['caption_length'] = val_df['structured_caption'].str.len()
test_df['caption_length'] = test_df['structured_caption'].str.len()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

sns.histplot(train_df['caption_length'], ax=axes[0], bins=30)
axes[0].set_title('Train - Caption Length')
axes[0].set_xlabel('Character Count')

sns.histplot(val_df['caption_length'], ax=axes[1], bins=30)
axes[1].set_title('Validation - Caption Length')
axes[1].set_xlabel('Character Count')

sns.histplot(test_df['caption_length'], ax=axes[2], bins=30)
axes[2].set_title('Test - Caption Length')
axes[2].set_xlabel('Character Count')

plt.tight_layout()
plt.show()

## Sample Images

In [ ]:
# Display sample X-ray images
sample_images = test_df.sample(6, random_state=42)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for i, (idx, row) in enumerate(sample_images.iterrows()):
    try:
        img = Image.open(row['image_path'])
        axes[i].imshow(img, cmap='gray')
        axes[i].set_title(f"Sample {i+1}\n{row['structured_caption'][:50]}...", fontsize=10)
        axes[i].axis('off')
    except Exception as e:
        axes[i].text(0.5, 0.5, f"Error loading image\n{str(e)}", 
                    ha='center', va='center', transform=axes[i].transAxes)
        axes[i].axis('off')

plt.tight_layout()
plt.show()

## Clinical Entities Analysis

In [ ]:
# Extract clinical entities from captions
clinical_keywords = [
    "pneumonia", "opacity", "effusion", "edema", "atelectasis",
    "cardiomegaly", "pleural", "pulmonary", "thorax", "lung",
    "consolidation", "infiltrate", "pneumothorax", "fibrosis",
    "nodule", "mass", "emphysema"
]

def count_entities(caption):
    caption_lower = caption.lower()
    return sum(1 for keyword in clinical_keywords if keyword in caption_lower)

train_df['entity_count'] = train_df['structured_caption'].apply(count_entities)
val_df['entity_count'] = val_df['structured_caption'].apply(count_entities)
test_df['entity_count'] = test_df['structured_caption'].apply(count_entities)

# Entity count distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

sns.countplot(x='entity_count', data=train_df, ax=axes[0])
axes[0].set_title('Train - Entity Count')

sns.countplot(x='entity_count', data=val_df, ax=axes[1])
axes[1].set_title('Validation - Entity Count')

sns.countplot(x='entity_count', data=test_df, ax=axes[2])
axes[2].set_title('Test - Entity Count')

plt.tight_layout()
plt.show()

## Summary

This notebook provides:
- Dataset size and split information
- Text length analysis across splits
- Sample X-ray images with captions
- Clinical entity distribution

Next steps: Proceed to model training and retrieval evaluation.